# How to build a turbo spin echo today

Two notebooks in this directory build the same sequence, and they are not alternatives:

| | |
|---|---|
| [`01_build.ipynb`](01_build.ipynb) | the **historical / reference implementation**, kept on purpose. It is the working code `TSEShot` and `FSE2D` were extracted from, and CI compares the package against it event for event. It must stay able to *disagree* with the package, so it is not rewritten to import one |
| **this notebook** | the **recommended way to build an FSE today**, using the shipped SeqCraft Modules |

The package ships two classes:

| | |
|---|---|
| `sc.modules.TSEShot` | the **kernel** — one excitation and its echo train. Owns the crusher window three axes share, the echo spacing it implies, and the placement that keeps every echo at the midpoint between its two refocusing pulses |
| `sc.modules.FSE2D` | the **imaging** module — how many shots, which lines each acquires, how many dummies, and therefore the effective TE |

`echoes=1` is a conventional spin echo and one long train with `partial_fourier` below 1 is
HASTE. Neither needs a class of its own.

In [ ]:
import numpy as np
import pypulseq as pp

import seqcraft as sc

opts = pp.Opts(
    max_grad=32, grad_unit='mT/m',
    max_slew=130, slew_unit='T/m/s',
    B0=3.0,
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
)

FOV_MM, MATRIX, THICKNESS_MM = (256.0, 256.0), (128, 128), 5.0
ECHOES, TR_S = 16, 2.0
NY = MATRIX[1]

## The kernel: one shot

`TSEShot` is the repeating unit. Everything it reports is derived from the geometry — you ask for
a turbo factor and a protocol, and it tells you what echo spacing that implies and **which of
three minima set it**.

In [ ]:
shot = sc.modules.TSEShot(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
                          echoes=ECHOES, tr_s=TR_S)

print(f'echo spacing {shot.echo_spacing_s * 1e3:7.3f} ms   ({shot.bound}-bound)')
print(f'crusher window {shot.crush_s * 1e6:5.0f} us from the three axis minima '
      + ', '.join(f'{axis} {value * 1e6:.0f} us' for axis, value in shot.crush_min_s.items()))
print(f'TE of echo n  ' + ', '.join(f'{shot.te_s(n) * 1e3:.1f}' for n in range(4)) + ' ... ms')
print(f'shot occupies {shot.shot_s * 1e3:7.3f} ms, TR {shot.tr_s * 1e3:.0f} ms '
      f'(minimum {shot.min_tr_s * 1e3:.1f} ms)')

One shot is a `LogicBlock`, and it takes **one line per echo** — which lines is not its business:

In [ ]:
one = shot(lines=[n * 8 for n in range(ECHOES)])
print(f'{one.duration * 1e3:.0f} ms, exactly TR')

k = sc.kspace(one, opts)
at_echo = k['k_adc'].reshape(3, ECHOES, shot.ro.num_samples)[:, :, shot.ro.echo_sample(0)]
spacing = np.diff(k['t_refocusing'])
print(f'|kx| at the echoes  {np.abs(at_echo[0]).max():.2e} 1/m')
print(f'|kz| at the echoes  {np.abs(at_echo[2]).max():.2e} 1/m')
print(f'refocusing spacing spread {np.ptp(spacing) * 1e12:.2f} ps')

## The imaging layer: which lines, in what order

`FSE2D` builds a `TSEShot` from the same arguments and adds the one thing a shot cannot know —
the table. `segments` is a list of line lists, one per shot, and that single argument carries the
turbo factor, the ordering **and** the effective TE.

No generator ships for it. `writeTSE.m`'s own reordering is commented *"TSE echo time magic"*,
which is a protocol choice rather than arithmetic.

In [ ]:
fse = sc.modules.FSE2D(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
                       echoes=ECHOES, tr_s=TR_S)

shots = NY // ECHOES
interleaved = [[s + n * shots for n in range(ECHOES)] for s in range(shots)]
linear = [list(range(s * ECHOES, (s + 1) * ECHOES)) for s in range(shots)]
centric_order = sorted(range(NY), key=lambda line: (abs(line - NY // 2), line))
centric = [[centric_order[n * shots + s] for n in range(ECHOES)] for s in range(shots)]

for name, table in (('interleaved', interleaved), ('centric', centric)):
    where = fse.echo_of_center_line(table)
    print(f'{name:12s} k=0 at shot {where[0]}, echo {where[1]:2d}  ->  '
          f'te_eff {fse.te_eff_s(table) * 1e3:6.1f} ms')

Same 128 lines, same waveforms, different contrast. `linear` is the table the band warning is
about: each echo index becomes a *comb* across k-space, and a periodic modulation of k-space is a
replica of the object rather than a blur.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', sc.SeqCraftWarning)
    fse(segments=linear)
print('linear ->', str(caught[0].message).split('.')[0] if caught else 'no warning')

bands = fse.echo_bands(interleaved)
print(f'interleaved widest echo band: {max(bands.values())} lines')

### The whole scan

`dummy_shots` is not decoration: shot 0 starts from thermal equilibrium and every later shot
starts from whatever the last TR recovered to, and with interleaved segmentation that one
difference is a comb in k-space.

In [ ]:
scan = fse(segments=interleaved, dummy_shots=1)
seq = sc.compile(scan, opts, name='fse_2d', definitions={
    'FOV': [FOV_MM[0] / 1e3, FOV_MM[1] / 1e3, THICKNESS_MM / 1e3],
    'kSpaceCenterLine': fse.center_line,
    'TE': fse.te_eff_s(interleaved),
    'TR': fse.shot.tr_s,
    'EchoSpacing': fse.shot.echo_spacing_s,
    'TurboFactor': fse.echoes,
})
print(f'{len(interleaved)} shots + 1 dummy, {len(seq.block_events)} blocks, '
      f'{seq.duration()[0]:.0f} s')

## HASTE is a configuration

One shot, a long train, and partial Fourier on the readout. Nothing new is constructed.

In [ ]:
haste = sc.modules.FSE2D(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, thickness_mm=THICKNESS_MM,
                         echoes=72, partial_fourier=0.625, tr_s=TR_S)
table = [[NY // 2 - 8 + n for n in range(72)]]

print(f'{haste.shot.ro.num_samples} of {MATRIX[0]} samples, '
      f'{haste.shot.ro.echo_sample(0)} before the echo')
print(f'esp {haste.shot.echo_spacing_s * 1e3:.3f} ms ({haste.shot.bound}-bound), '
      f'te_eff {haste.te_eff_s(table) * 1e3:.1f} ms, '
      f'last echo at {haste.shot.te_s(71) * 1e3:.0f} ms')
print(f'the shot occupies {haste.shot.shot_s:.2f} s -- one breath hold; the block is\n'
      f'padded to the {haste.shot.tr_s:.1f} s TR, which a single-shot protocol does not need')

## What ships, and what does not

| | |
|---|---|
| `sc.modules.TSEShot` | the reusable shot — use it when you want to place the train yourself |
| `sc.modules.FSE2D` | the whole acquisition — use it when you want a scan |
| `01_build.ipynb`'s own `FSE2D` | **reference evidence**, kept so the package can be checked against the implementation it came from. It is intentionally *able to disagree* with the package; do not replace it with an import |

[`02_simulate_and_reconstruct.ipynb`](02_simulate_and_reconstruct.ipynb) measures what the train
costs: the point-spread width per ordering, the ghost a scattered table makes, and the echo
envelope.